# Step-by-step Sentinel-2 extraction
This notebook reproduces the pipeline from `01-sentinel2.ipynb`, but we instantiate each pipeline object explicitly and inspect the inputs and outputs of every stage.

Pipeline stages:
1. **SearchProvider** — discover STAC assets
2. **TaskBuilder** — group assets into extraction tasks
3. **Reader** — load raw data from a list of files
4. **Writer** — write a GeoTIFF to a given path
We also show how to run a single task with `run_task` and how to execute many tasks in parallel with `LocalExecutor`.


In [ ]:
from datetime import datetime, timezone
from functools import partial

from aereo.builtins.search import search_stac
from aereo.builtins.task_builder import build_grouped_tasks
from aereo.cache import TaskResultCache
from aereo.executors import LocalExecutor
from aereo.pipeline import ExtractionJob

# Load the same Hydra config package used by 01-sentinel2.ipynb.
job = ExtractionJob.load_from_config(
    config_dir="config",
    config_name="job_sentinel2",
)

# Search provider and task builder are plain Python callables.
# We bind the fixed parameters here so the later cells match the plugin style.
search_provider = partial(
    search_stac,
    stac_api_url="https://earth-search.aws.element84.com/v1",
    collections={"sentinel-2-l2a": ["red", "nir"]},
    intersects="config/aoi/chocon.geojson",
    start_datetime=datetime(2024, 1, 1, tzinfo=timezone.utc),
    end_datetime=datetime(2024, 1, 10, 23, 59, 59, tzinfo=timezone.utc),
)

task_builder = partial(build_grouped_tasks, cells_per_task=50)

print("Job name:", job.name)
print("Grid dist:", job.grid_dist)
print("Output URI:", job.output_uri)
print("Search provider callable:", callable(search_provider))
print("Task builder callable:", callable(task_builder))
print("Reader callable:", callable(job.read))
print("Writer callable:", callable(job.write))

## Step 1 — Search: `SearchSTAC`

The search provider queries the STAC API and returns a GeoDataFrame of matched assets. Each row corresponds to one requested asset (e.g. `red`, `nir`) from one STAC item.

In [ ]:
# The search_provider variable is a partial of aereo.builtins.search.search_stac.
# Run the search.
assets = search_provider()
print(f"✓ Found {len(assets)} asset rows")

# Inspect the output schema.
print("Columns:", list(assets.columns))
print("First rows:")
assets[["id", "collection", "channel_id", "crs", "href"]].head()

## Inspect the first task in detail


In [ ]:
# The task_builder variable is a partial of aereo.builtins.task_builder.build_grouped_tasks.
tasks = task_builder(assets, job)
print(f"✓ Built {len(tasks)} extraction task(s)")

for i, task in enumerate(tasks):
    print(f"Task {i}: {task}")

In [ ]:
# Inspect the first task in detail
task = tasks[0]

print("Grid cells:", task.grid_cells)
print("Task context:", dict(task.task_context))
print(f"Number of asset rows in task: {len(task.assets)}")

# Extraction stages available on the task (delegated from the job)
print("Extraction stages:")
print("  read:", task.read.__name__)
print("  write:", task.write.__name__)

## Step 3 — Read: `ReadODCSTAC`

The reader reconstructs `pystac.Item` objects from the `stac_item` column and uses `odc.stac.load` to build a lazy `xarray.Dataset` in the native CRS of the STAC items.

In [ ]:
from aereo.builtins.read import read_odc_stac

# Option A: use the reader attached to the job
reader = job.read

# Option B: use the function explicitly
explicit_reader = read_odc_stac

files = task.assets["href"].tolist()
ds_native = reader(
    files, assets=task.assets
)  # equivalent to explicit_reader(files, assets=task.assets)

print("Native dataset:")
print(ds_native)
print("Dataset attributes:")
print(ds_native.attrs)

In [ ]:
ds_native

## Step 4 — Write: `WriteGeoTIFF`

The writer serialises a dataset to a GeoTIFF at the supplied path and returns the written path.


In [ ]:
from pathlib import Path
from aereo.builtins.write import write_geotiff

# Option A: use the writer attached to the job
writer = job.write

# Option B: use the function explicitly
explicit_writer = write_geotiff

out_path = Path(job.output_uri) / "demo_native_red.tif"
written = writer(
    ds_native, out_path
)  # equivalent to explicit_writer(ds_native, out_path)

print(f"\u2713 Wrote {written}")

In [ ]:
import rioxarray

rioxarray.open_rasterio(written)[0].plot()

## Putting it together: `run_task`

`run_task(task)` executes the fixed pipeline `read → preprocess → reproject → postprocess → write` for a single `ExtractionTask`. It is what `LocalExecutor` uses under the hood.


In [ ]:
from aereo.execution import run_task

artifacts_gdf = run_task(task)

print(f"\u2713 Single task produced {len(artifacts_gdf)} artifact row(s)")
artifacts_gdf[["id", "uri", "grid_cell", "geometry"]].head()

In [ ]:
rioxarray.open_rasterio(artifacts_gdf.iloc[4].uri)[0].plot()

## Batch execution with an executor

To run many tasks in parallel, pass the task list to an `Executor`. For COG/I/O-bound extractors like this one, `LocalExecutor(use_threads=True)` is the safer choice in Jupyter and avoids the pickling issues that can make `ProcessPoolExecutor` hang.

In [ ]:
executor = LocalExecutor(workers=4, use_threads=True, cache=TaskResultCache())
artifacts = executor(tasks)

print(f"✓ Extracted {len(artifacts)} artifact row(s) from {len(tasks)} task(s)")
artifacts[["id", "uri", "grid_cell", "start_time"]].head()

In [ ]:
from aereo.viz import plot_artifact_patches

plot_artifact_patches(artifacts, ds_factor=10, cmap="viridis")